#### 01. Find the total matches played, total wins & total losses by each team

In [0]:
%sql
---- Creating new catalog, schema -----
create catalog if not exists sql_youtube_practise;
use catalog sql_youtube_practise;
create schema if not exists sql;
use sql;
show current schema;

catalog,namespace
sql_youtube_practise,sql


In [0]:
%sql
DROP TABLE IF EXISTS icc_world_cup;
create table icc_world_cup
(
Team_1 Varchar(20),
Team_2 Varchar(20),
Winner Varchar(20)
);
INSERT INTO icc_world_cup values('India','SL','India');
INSERT INTO icc_world_cup values('SL','Aus','Aus');
INSERT INTO icc_world_cup values('SA','Eng','Eng');
INSERT INTO icc_world_cup values('Eng','NZ','NZ');
INSERT INTO icc_world_cup values('Aus','India','India');
select * from icc_world_cup;

Team_1,Team_2,Winner
Aus,India,India
India,SL,India
SA,Eng,Eng
SL,Aus,Aus
Eng,NZ,NZ


In [0]:
%sql
with cte as ( 
    select
        Team_1 as team_name,
        case
            when Winner = Team_1 then 1
            else 0
        end as Wins
    from icc_world_cup
    union all
    select
        Team_2 as team_name,
        case
            when Winner = Team_2 then 1
            else 0
        end as Wins
    from icc_world_cup
)
select
    team_name,
    count(1) as total_matches_played,
    sum(Wins) as Wins,
    count(1) - sum(Wins) as Losses
from cte
group by team_name
order by Wins desc;

team_name,total_matches_played,Wins,Losses
India,2,2,0
Aus,2,1,1
Eng,2,1,1
NZ,1,1,0
SA,1,0,1
SL,2,0,2


#### 02. Find the number of first-time customers and repeat customers for each order date.

In [0]:
%sql
DROP TABLE IF EXISTS customer_orders;
CREATE TABLE customer_orders (
    order_id INT,
    customer_id INT,
    order_date DATE,
    order_amount INT
);
INSERT INTO customer_orders VALUES
(1,100,CAST('2022-01-01' AS DATE),2000),
(2,200,CAST('2022-01-01' AS DATE),2500),
(3,300,CAST('2022-01-01' AS DATE),2100),
(4,100,CAST('2022-01-02' AS DATE),2000),
(5,400,CAST('2022-01-02' AS DATE),2200),
(6,500,CAST('2022-01-02' AS DATE),2700),
(7,100,CAST('2022-01-03' AS DATE),3000),
(8,400,CAST('2022-01-03' AS DATE),1000),
(9,600,CAST('2022-01-03' AS DATE),3000);
SELECT * FROM customer_orders;

order_id,customer_id,order_date,order_amount
1,100,2022-01-01,2000
2,200,2022-01-01,2500
3,300,2022-01-01,2100
4,100,2022-01-02,2000
5,400,2022-01-02,2200
6,500,2022-01-02,2700
7,100,2022-01-03,3000
8,400,2022-01-03,1000
9,600,2022-01-03,3000


In [0]:
%sql
with cte as(
    select 
        customer_id,
        min(order_date) as first_visit_date
    from customer_orders
    group by customer_id
),
cte1 as(
    select
        co.*,
        cte.first_visit_date,
        case
            when co.order_date = cte.first_visit_date then 1
            else 0
        end as first_visit_flag,
        case
            when co.order_date != cte.first_visit_date then 1
            else 0
        end as repeat_visit_flag
    from customer_orders as co
    join cte 
    on cte.customer_id = co.customer_id
)
select
    cte1.order_date,
    sum(first_visit_flag) as first_visits,
    sum(repeat_visit_flag) as repeat_visits
from cte1
group by cte1.order_date
order by cte1.order_date;

order_date,first_visits,repeat_visits
2022-01-01,3,0
2022-01-02,2,1
2022-01-03,1,2


###### Using Windows Function

In [0]:
%sql
WITH cte AS (
    SELECT
        *,
        MIN(order_date) OVER (PARTITION BY customer_id) AS first_visit_date
    FROM customer_orders
)
SELECT
    order_date,
    SUM(CASE WHEN order_date = first_visit_date THEN 1 ELSE 0 END) AS first_visits,
    SUM(CASE WHEN order_date != first_visit_date THEN 1 ELSE 0 END) AS repeat_visits
FROM cte
GROUP BY order_date
ORDER BY order_date;

order_date,first_visits,repeat_visits
2022-01-01,3,0
2022-01-02,2,1
2022-01-03,1,2


##### 03. Find each employee's 
- most visited floor, 
- total number of visits, and
- distinct resources they used.

In [0]:
%sql
DROP TABLE IF EXISTS entries;
CREATE TABLE entries (
    name STRING,
    address STRING,
    email STRING,
    floor INT,
    resources STRING
);
INSERT INTO entries VALUES
('A','Bangalore','A@gmail.com',1,'CPU'),
('A','Bangalore','A1@gmail.com',1,'CPU'),
('A','Bangalore','A2@gmail.com',2,'DESKTOP'),
('B','Bangalore','B@gmail.com',2,'DESKTOP'),
('B','Bangalore','B1@gmail.com',2,'DESKTOP'),
('B','Bangalore','B2@gmail.com',1,'MONITOR');
SELECT * FROM entries;

name,address,email,floor,resources
A,Bangalore,A@gmail.com,1,CPU
A,Bangalore,A1@gmail.com,1,CPU
A,Bangalore,A2@gmail.com,2,DESKTOP
B,Bangalore,B@gmail.com,2,DESKTOP
B,Bangalore,B1@gmail.com,2,DESKTOP
B,Bangalore,B2@gmail.com,1,MONITOR


In [0]:
%sql
with cte1 as (
    select
        name,
        count(floor) as total_visits,
        string_agg(distinct resources, ',') as resources_used
    from entries 
    group by name
),
cte as (
    select
        name,
        floor,
        count(floor) as no_of_floor_visit,
        dense_rank() over(partition by name order by count(floor) desc) as rank
    from entries
    group by name, floor
)
select
    cte.name,
    cte1.total_visits,
    cte.floor as most_visited_floor,
    cte1.resources_used
from cte
inner join cte1
    on cte.name = cte1.name
where rank = 1;


name,total_visits,most_visited_floor,resources_used
B,3,2,"MONITOR,DESKTOP"
A,3,1,"CPU,DESKTOP"


##### 04. Find the person IDs and names of all users whose friends' total score is greater than 100, along with the number of friends and their total score.

In [0]:
%sql
DROP TABLE IF EXISTS person;
CREATE TABLE person (
    PersonID INT,
    Name STRING,
    Email STRING,
    Score INT
);
INSERT INTO person VALUES
(1,'Alice','alice2018@hotmail.com',88),
(2,'Bob','bob2018@hotmail.com',11),
(3,'Davis','davis2018@hotmail.com',27),
(4,'Tara','tara2018@hotmail.com',45),
(5,'John','john2018@hotmail.com',63);

DROP TABLE IF EXISTS friend;
CREATE TABLE friend (
    PersonID INT,
    FriendID INT
);

INSERT INTO friend VALUES
(1,2),
(1,3),
(2,1),
(2,3),
(3,5),
(4,2),
(4,3),
(4,5);

SELECT * FROM person;
SELECT * FROM friend;

PersonID,FriendID
1,2
1,3
2,1
2,3
3,5
4,2
4,3
4,5


In [0]:
%sql
SELECT * FROM person;

PersonID,Name,Email,Score
1,Alice,alice2018@hotmail.com,88
2,Bob,bob2018@hotmail.com,11
3,Davis,davis2018@hotmail.com,27
4,Tara,tara2018@hotmail.com,45
5,John,john2018@hotmail.com,63


In [0]:
%sql
with cte as (
    select
        f.PersonID,
        f.FriendID,
        p.Score
    from friend as f
    join person as p 
    on f.FriendID = p.PersonID
),
cte1 as (
    select
        PersonID,
        count(FriendID) as total_friends,
        sum(Score) as total_friends_score
    from cte
    group by PersonID
    having sum(Score) > 100
)
select 
    p.PersonID,
    p.Name,
    cte1.total_friends,
    cte1.total_friends_score
from person as p
join cte1
    on p.PersonID = cte1.PersonID;



PersonID,Name,total_friends,total_friends_score
2,Bob,2,115
4,Tara,3,101


##### 05. Calculate the daily cancellation percentage for trips where both the client and the driver are not banned.

###### Concept: Filter valid trips, count cancelled and total trips for each day, then calculate the cancellation percentage.

Formula:
Cancellation Rate (%) = (Cancelled Trips / Total Trips) × 100

In [0]:
%sql
DROP TABLE IF EXISTS Trips;
DROP TABLE IF EXISTS Users;
CREATE TABLE Trips (
    id INT,
    client_id INT,
    driver_id INT,
    city_id INT,
    status STRING,
    request_at DATE
);
CREATE TABLE Users (
    users_id INT,
    banned STRING,
    role STRING
);
INSERT INTO Trips VALUES
(1,1,10,1,'completed','2013-10-01'),
(2,2,11,1,'cancelled_by_driver','2013-10-01'),
(3,3,12,6,'completed','2013-10-01'),
(4,4,13,6,'cancelled_by_client','2013-10-01'),
(5,1,10,1,'completed','2013-10-02'),
(6,2,11,6,'completed','2013-10-02'),
(7,3,12,6,'completed','2013-10-02'),
(8,2,12,12,'completed','2013-10-03'),
(9,3,10,12,'completed','2013-10-03'),
(10,4,13,12,'cancelled_by_driver','2013-10-03');
INSERT INTO Users VALUES
(1,'No','client'),
(2,'Yes','client'),
(3,'No','client'),
(4,'No','client'),
(10,'No','driver'),
(11,'No','driver'),
(12,'No','driver'),
(13,'No','driver');

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
select * from trips;

id,client_id,driver_id,city_id,status,request_at
1,1,10,1,completed,2013-10-01
2,2,11,1,cancelled_by_driver,2013-10-01
3,3,12,6,completed,2013-10-01
4,4,13,6,cancelled_by_client,2013-10-01
5,1,10,1,completed,2013-10-02
6,2,11,6,completed,2013-10-02
7,3,12,6,completed,2013-10-02
8,2,12,12,completed,2013-10-03
9,3,10,12,completed,2013-10-03
10,4,13,12,cancelled_by_driver,2013-10-03


In [0]:
%sql
select * from users;

users_id,banned,role
1,No,client
2,Yes,client
3,No,client
4,No,client
10,No,driver
11,No,driver
12,No,driver
13,No,driver


In [0]:
%sql
select *
from trips t
join users u on t.client_id = u.users_id
join users u1 on t.driver_id = u1.users_id
where u.banned = "No" and u1.banned = "No";

id,client_id,driver_id,city_id,status,request_at,users_id,banned,role,users_id,banned,role
1,1,10,1,completed,2013-10-01,1,No,client,10,No,driver
3,3,12,6,completed,2013-10-01,3,No,client,12,No,driver
4,4,13,6,cancelled_by_client,2013-10-01,4,No,client,13,No,driver
5,1,10,1,completed,2013-10-02,1,No,client,10,No,driver
7,3,12,6,completed,2013-10-02,3,No,client,12,No,driver
9,3,10,12,completed,2013-10-03,3,No,client,10,No,driver
10,4,13,12,cancelled_by_driver,2013-10-03,4,No,client,13,No,driver


In [0]:
%sql
with cte as (
    select
        t.request_at,
        count(
            case
                when t.status != 'completed' then 1 else null
            end
        ) as cancelled_trips,
        count(1) as total_trips
    from trips t
    join users u
        on t.client_id = u.users_id
    join users u1
        on t.driver_id = u1.users_id
    where u.banned = "No" and u1.banned = "No"
    group by t.request_at
    order by t.request_at asc
)
select
    request_at,
    cancelled_trips,
    total_trips,
    round(cancelled_trips * 100 / total_trips, 2) as cancellation_rate
from cte;

request_at,cancelled_trips,total_trips,cancellation_rate
2013-10-01,1,3,33.33
2013-10-02,0,2,0.0
2013-10-03,1,2,50.0
